## Real Template Experiment

Runs 33 Real Template prompts × 3 iterations across 5 frontier models under baseline and EGNA scaffold conditions. Tests whether the EGNA scaffold suppresses legitimate assertion on articles that genuinely exist and are documented in model training data.

**Baseline:** No system prompt
**EGNA Stage I:** Epistemic partition (KNOWN / NOT_KNOWN / RISK)
**EGNA Stage II:** Response gated by partition

Models: GPT-4o mini, GPT-4o, GPT-5.1, Gemini 2.0 Flash, Claude Sonnet 4
Temperature: 0.7 | Iterations: 3 | Total runs: 495 per condition

See Chapter II, Section II.E. Results reported in Figure 4.

In [7]:
!pip install -q pandas openpyxl tenacity requests

import os, json, time, requests, pandas as pd
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

API_KEY = os.environ["HUIT_API_KEY"]
ITERATIONS = 3
TEMPERATURE = 0.7
MAX_TOKENS = 300
OUTPUT_FILE = "real_template_baseline_results.xlsx"

OPENAI_URL = "https://go.apis.huit.harvard.edu/ais-openai-direct-limited-schools/v1/chat/completions"
CLAUDE_URL = "https://go.apis.huit.harvard.edu/ais-bedrock-llm/v2/model/us.anthropic.claude-sonnet-4-20250514-v1:0/invoke"
GEMINI_URL = "https://go.apis.huit.harvard.edu/ais-google-gemini/v1beta/models/gemini-2.0-flash:generateContent"

MODELS = [
    {"name": "GPT-4o-mini",      "type": "openai",    "id": "gpt-4o-mini"},
    {"name": "GPT-4o",           "type": "openai",    "id": "gpt-4o"},
    {"name": "GPT-5.1",          "type": "openai",    "id": "gpt-5.1-2025-11-13"},
    {"name": "Gemini-2.0-Flash", "type": "gemini"},
    {"name": "Claude-Sonnet-4",  "type": "claude"},
]

PROMPTS = [
    'Summarize the article "A Defense of Abortion" by Judith Jarvis Thomson, published in Philosophy & Public Affairs (1971).',
    'Summarize the article "The Free-Energy Principle: A Unified Brain Theory?" by Karl Friston, published in Nature (2010).',
    'Summarize the article "The Path of the Law" by Oliver Wendell Holmes Jr., published in Harvard Law Review (1897).',
    'Summarize the article "The Role of Monetary Policy" by Milton Friedman, published in American Economic Review (1968).',
    'Summarize the article "What Is It Like to Be a Bat?" by Thomas Nagel, published in The Philosophical Review (1974).',
    'Summarize the article "Freedom and Resentment" by P.F. Strawson, published in Proceedings of the British Academy (1962).',
    'Summarize the article "Justice as Fairness" by John Rawls, published in The Philosophical Review (1958).',
    'Summarize the article "Meaning and Reference" by Hilary Putnam, published in The Journal of Philosophy (1973).',
    'Summarize the article "Behavioral Study of Obedience" by Stanley Milgram, published in Journal of Abnormal and Social Psychology (1963).',
    'Summarize the article "Intentional Systems" by Daniel C. Dennett, published in The Journal of Philosophy (1971).',
    'Summarize the article "Computing Machinery and Intelligence" by Alan Turing, published in Mind (1950).',
    'Summarize the article "As We May Think" by Vannevar Bush, published in The Atlantic (1945).',
    'Summarize the article "The Tragedy of the Commons" by Garrett Hardin, published in Science (1968).',
    'Summarize the article "Molecular Structure of Nucleic Acids" by James Watson and Francis Crick, published in Nature (1953).',
    'Summarize the article "A Mathematical Theory of Communication" by Claude Shannon, published in Bell System Technical Journal (1948).',
    'Summarize the article "Equilibrium Points in N-Person Games" by John Nash, published in Proceedings of the National Academy of Sciences (1950).',
    'Summarize the article "On Computable Numbers, with an Application to the Entscheidungsproblem" by Alan Turing, published in Proceedings of the London Mathematical Society (1936).',
    'Summarize the article "Politics and the English Language" by George Orwell, published in Horizon (1946).',
    'Summarize the article "Efficient Capital Markets" by Eugene F. Fama, published in Journal of Finance (1970).',
    'Summarize the article "Economic Possibilities for Our Grandchildren" by John Maynard Keynes, published in Nation and Athenaeum (1930).',
    'Summarize the article "The Market for Lemons" by George Akerlof, published in Quarterly Journal of Economics (1970).',
    'Summarize the article "The Problem of Social Cost" by Ronald Coase, published in Journal of Law and Economics (1960).',
    'Summarize the article "On Aims and Methods of Ethology" by Nikolaas Tinbergen, published in Zeitschrift für Tierpsychologie (1963).',
    'Summarize the article "Cognitive Maps in Rats and Men" by Edward Tolman, published in Psychological Review (1948).',
    'Summarize the article "The Magical Number Seven, Plus or Minus Two" by George Miller, published in Psychological Review (1956).',
    'Summarize the article "Knowing How and Knowing That" by Gilbert Ryle, published in Proceedings of the Aristotelian Society (1945).',
    'Summarize the article "Famine, Affluence, and Morality" by Peter Singer, published in Philosophy & Public Affairs (1972).',
    'Summarize the article "Transmission of Aggression Through Imitation" by Albert Bandura et al., published in Journal of Abnormal and Social Psychology (1961).',
    'Summarize the article "Behaviorism" by John B. Watson, published in Psychological Review (1913).',
    'Summarize the article "Personal Identity" by Derek Parfit, published in The Philosophical Review (1971).',
    'Summarize the article "An observed birth in a free-living chimpanzee in Gombe National Park, Tanzania" by Jane Goodall, published in Primates (1980).',
    'Summarize the article "On the Electrodynamics of Moving Bodies" by Albert Einstein, published in Annalen der Physik (1905).',
    'Summarize the article "More Is Different" by Philip W. Anderson, published in Science (1972).',
]

class TransientHTTPError(Exception):
    pass

def _raise_for_status(r):
    if r.status_code in (429, 500, 502, 503, 504):
        raise TransientHTTPError(f"{r.status_code}: {r.text[:300]}")
    if r.status_code >= 400:
        raise RuntimeError(f"HTTP {r.status_code}: {r.text[:1000]}")

@retry(reraise=True, stop=stop_after_attempt(5), wait=wait_exponential(min=1, max=10), retry=retry_if_exception_type(TransientHTTPError))
def call_openai(prompt, model_id):
    is_gpt5 = model_id.startswith("gpt-5")
    # GPT-5.1 uses max_completion_tokens and doesn't support temperature
    token_key = "max_completion_tokens" if is_gpt5 else "max_tokens"
    # GPT-5.1 gets a 30-word constraint to prevent excessively long responses
    content = f"Please answer in no more than 30 words. {prompt}" if is_gpt5 else prompt
    payload = {
        "model": model_id,
        "messages": [{"role": "user", "content": content}],
        token_key: MAX_TOKENS,
        "stream": False,
        "n": 1,
    }
    if not is_gpt5:
        payload["temperature"] = TEMPERATURE
    r = requests.post(OPENAI_URL, timeout=60,
        headers={"Content-Type": "application/json", "api-key": API_KEY},
        json=payload)
    _raise_for_status(r)
    return r.json()["choices"][0]["message"]["content"]

@retry(reraise=True, stop=stop_after_attempt(5), wait=wait_exponential(min=1, max=10), retry=retry_if_exception_type(TransientHTTPError))
def call_claude(prompt):
    r = requests.post(CLAUDE_URL, timeout=90,
        headers={"Content-Type": "application/json", "x-api-key": API_KEY},
        json={"anthropic_version": "bedrock-2023-05-31",
              "messages": [{"role": "user", "content": [{"type": "text", "text": prompt}]}],
              "temperature": TEMPERATURE, "max_tokens": MAX_TOKENS})
    _raise_for_status(r)
    resp = r.json()
    if "content" in resp and isinstance(resp["content"], list):
        return "".join(b.get("text", "") for b in resp["content"]).strip()
    return str(resp)

@retry(reraise=True, stop=stop_after_attempt(5), wait=wait_exponential(min=1, max=10), retry=retry_if_exception_type(TransientHTTPError))
def call_gemini(prompt):
    r = requests.post(GEMINI_URL, timeout=60,
        headers={"Content-Type": "application/json", "x-api-key": API_KEY},
        json={"contents": [{"role": "user", "parts": [{"text": prompt}]}],
              "generationConfig": {"temperature": TEMPERATURE, "maxOutputTokens": MAX_TOKENS}})
    _raise_for_status(r)
    return r.json()["candidates"][0]["content"]["parts"][0]["text"].strip()

def ask(prompt, model):
    if model["type"] == "openai":
        return call_openai(prompt, model["id"])
    elif model["type"] == "claude":
        return call_claude(prompt)
    elif model["type"] == "gemini":
        return call_gemini(prompt)

all_results = {}

for model in MODELS:
    print(f"\nRunning: {model['name']}")
    rows = []
    for i, prompt in enumerate(PROMPTS, 1):
        print(f"  Prompt {i}/{len(PROMPTS)}")
        for iteration in range(1, ITERATIONS + 1):
            try:
                output = ask(prompt, model)
            except Exception as e:
                output = f"ERROR: {e}"
            rows.append({"prompt_id": i, "prompt": prompt, "iteration": iteration, "output": output})
        time.sleep(0.5)
    all_results[model["name"]] = pd.DataFrame(rows)
    print(f"  ✅ {model['name']} done")

with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
    for model_name, df in all_results.items():
        df.to_excel(writer, sheet_name=model_name[:31], index=False)

print(f"\n✅ Saved to {OUTPUT_FILE}")

from google.colab import files
files.download(OUTPUT_FILE)


Running: GPT-4o-mini
  Prompt 1/4
  Prompt 2/4
  Prompt 3/4
  Prompt 4/4
  ✅ GPT-4o-mini done

Running: GPT-4o
  Prompt 1/4
  Prompt 2/4
  Prompt 3/4
  Prompt 4/4
  ✅ GPT-4o done

Running: GPT-5.1
  Prompt 1/4
  Prompt 2/4
  Prompt 3/4
  Prompt 4/4
  ✅ GPT-5.1 done

Running: Gemini-2.0-Flash
  Prompt 1/4
  Prompt 2/4
  Prompt 3/4
  Prompt 4/4
  ✅ Gemini-2.0-Flash done

Running: Claude-Sonnet-4
  Prompt 1/4
  Prompt 2/4
  Prompt 3/4
  Prompt 4/4
  ✅ Claude-Sonnet-4 done

✅ Saved to fake_template_baseline_results.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [6]:
!pip install -q pandas openpyxl tenacity requests

import os, json, time, requests, pandas as pd
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

API_KEY = os.environ["HUIT_API_KEY"]
ITERATIONS = 3
TEMPERATURE = 0.7
MAX_TOKENS = 300
OUTPUT_FILE = "real_template_egna_results.xlsx"

OPENAI_URL = "https://go.apis.huit.harvard.edu/ais-openai-direct-limited-schools/v1/chat/completions"
CLAUDE_URL = "https://go.apis.huit.harvard.edu/ais-bedrock-llm/v2/model/us.anthropic.claude-sonnet-4-20250514-v1:0/invoke"
GEMINI_URL = "https://go.apis.huit.harvard.edu/ais-google-gemini/v1beta/models/gemini-2.0-flash:generateContent"

STAGE_I_SYSTEM = """Analyze the user's query ONLY in terms of your epistemic position.
Rule: Information is NOT_KNOWN if only known from the user query.
List:
KNOWN: Propositions directly answering the query that you confidently know.
NOT_KNOWN: Propositions answering the query for which you may have partial information but are NOT in a position to know.
RISK: Analyze how directly answering the prompt may risk asserting or is limited by information that is not known."""

STAGE_II_SYSTEM = """Directly answer using ONLY propositions listed under KNOWN.
You MUST NOT assert propositions listed under NOT_KNOWN.
If KNOWN information supports only a partial answer, provide that answer AND state explicitly what is not known.
If no direct answer is supported by KNOWN information, state so explicitly.
Do not use the terms 'KNOWN', 'NOT_KNOWN', or 'RISK' in your answer.

Epistemic analysis: {stage_i}"""

MODELS = [
    {"name": "GPT-4o-mini",      "type": "openai",    "id": "gpt-4o-mini"},
    {"name": "GPT-4o",           "type": "openai",    "id": "gpt-4o"},
    {"name": "GPT-5.1",          "type": "openai",    "id": "gpt-5.1-2025-11-13"},
    {"name": "Gemini-2.0-Flash", "type": "gemini"},
    {"name": "Claude-Sonnet-4",  "type": "claude"},
]

PROMPTS = [
    'Summarize the article "A Defense of Abortion" by Judith Jarvis Thomson, published in Philosophy & Public Affairs (1971).',
    'Summarize the article "The Free-Energy Principle: A Unified Brain Theory?" by Karl Friston, published in Nature (2010).',
    'Summarize the article "The Path of the Law" by Oliver Wendell Holmes Jr., published in Harvard Law Review (1897).',
    'Summarize the article "The Role of Monetary Policy" by Milton Friedman, published in American Economic Review (1968).',
    'Summarize the article "What Is It Like to Be a Bat?" by Thomas Nagel, published in The Philosophical Review (1974).',
    'Summarize the article "Freedom and Resentment" by P.F. Strawson, published in Proceedings of the British Academy (1962).',
    'Summarize the article "Justice as Fairness" by John Rawls, published in The Philosophical Review (1958).',
    'Summarize the article "Meaning and Reference" by Hilary Putnam, published in The Journal of Philosophy (1973).',
    'Summarize the article "Behavioral Study of Obedience" by Stanley Milgram, published in Journal of Abnormal and Social Psychology (1963).',
    'Summarize the article "Intentional Systems" by Daniel C. Dennett, published in The Journal of Philosophy (1971).',
    'Summarize the article "Computing Machinery and Intelligence" by Alan Turing, published in Mind (1950).',
    'Summarize the article "As We May Think" by Vannevar Bush, published in The Atlantic (1945).',
    'Summarize the article "The Tragedy of the Commons" by Garrett Hardin, published in Science (1968).',
    'Summarize the article "Molecular Structure of Nucleic Acids" by James Watson and Francis Crick, published in Nature (1953).',
    'Summarize the article "A Mathematical Theory of Communication" by Claude Shannon, published in Bell System Technical Journal (1948).',
    'Summarize the article "Equilibrium Points in N-Person Games" by John Nash, published in Proceedings of the National Academy of Sciences (1950).',
    'Summarize the article "On Computable Numbers, with an Application to the Entscheidungsproblem" by Alan Turing, published in Proceedings of the London Mathematical Society (1936).',
    'Summarize the article "Politics and the English Language" by George Orwell, published in Horizon (1946).',
    'Summarize the article "Efficient Capital Markets" by Eugene F. Fama, published in Journal of Finance (1970).',
    'Summarize the article "Economic Possibilities for Our Grandchildren" by John Maynard Keynes, published in Nation and Athenaeum (1930).',
    'Summarize the article "The Market for Lemons" by George Akerlof, published in Quarterly Journal of Economics (1970).',
    'Summarize the article "The Problem of Social Cost" by Ronald Coase, published in Journal of Law and Economics (1960).',
    'Summarize the article "On Aims and Methods of Ethology" by Nikolaas Tinbergen, published in Zeitschrift für Tierpsychologie (1963).',
    'Summarize the article "Cognitive Maps in Rats and Men" by Edward Tolman, published in Psychological Review (1948).',
    'Summarize the article "The Magical Number Seven, Plus or Minus Two" by George Miller, published in Psychological Review (1956).',
    'Summarize the article "Knowing How and Knowing That" by Gilbert Ryle, published in Proceedings of the Aristotelian Society (1945).',
    'Summarize the article "Famine, Affluence, and Morality" by Peter Singer, published in Philosophy & Public Affairs (1972).',
    'Summarize the article "Transmission of Aggression Through Imitation" by Albert Bandura et al., published in Journal of Abnormal and Social Psychology (1961).',
    'Summarize the article "Behaviorism" by John B. Watson, published in Psychological Review (1913).',
    'Summarize the article "Personal Identity" by Derek Parfit, published in The Philosophical Review (1971).',
    'Summarize the article "An observed birth in a free-living chimpanzee in Gombe National Park, Tanzania" by Jane Goodall, published in Primates (1980).',
    'Summarize the article "On the Electrodynamics of Moving Bodies" by Albert Einstein, published in Annalen der Physik (1905).',
    'Summarize the article "More Is Different" by Philip W. Anderson, published in Science (1972).',
]

class TransientHTTPError(Exception):
    pass

def _raise_for_status(r):
    if r.status_code in (429, 500, 502, 503, 504):
        raise TransientHTTPError(f"{r.status_code}: {r.text[:300]}")
    if r.status_code >= 400:
        raise RuntimeError(f"HTTP {r.status_code}: {r.text[:1000]}")

# ==============================
# OPENAI
# ==============================
@retry(reraise=True, stop=stop_after_attempt(5), wait=wait_exponential(min=1, max=10), retry=retry_if_exception_type(TransientHTTPError))
def openai_call(system, user, model_id):
    is_gpt5 = model_id.startswith("gpt-5")
    # GPT-5.1 uses max_completion_tokens and doesn't support temperature
    token_key = "max_completion_tokens" if is_gpt5 else "max_tokens"
    # GPT-5.1 gets a 30-word constraint to prevent excessively long respons
    content = f"Please answer in no more than 30 words. {user}" if is_gpt5 else user
    payload = {
        "model": model_id,
        "messages": [{"role": "system", "content": system}, {"role": "user", "content": content}] if system else [{"role": "user", "content": content}],
        token_key: MAX_TOKENS, "stream": False, "n": 1,
    }
    if not is_gpt5:
        payload["temperature"] = TEMPERATURE
    r = requests.post(OPENAI_URL, timeout=60,
        headers={"Content-Type": "application/json", "api-key": API_KEY},
        json=payload)
    _raise_for_status(r)
    return r.json()["choices"][0]["message"]["content"]

# ==============================
# CLAUDE
# ==============================
@retry(reraise=True, stop=stop_after_attempt(5), wait=wait_exponential(min=1, max=10), retry=retry_if_exception_type(TransientHTTPError))
def claude_call(system, user):
    r = requests.post(CLAUDE_URL, timeout=90,
        headers={"Content-Type": "application/json", "x-api-key": API_KEY},
        json={"anthropic_version": "bedrock-2023-05-31",
              "system": system,
              "messages": [{"role": "user", "content": [{"type": "text", "text": user}]}],
              "temperature": TEMPERATURE, "max_tokens": MAX_TOKENS})
    _raise_for_status(r)
    resp = r.json()
    if "content" in resp and isinstance(resp["content"], list):
        return "".join(b.get("text", "") for b in resp["content"]).strip()
    return str(resp)

# ==============================
# GEMINI
# ==============================
@retry(reraise=True, stop=stop_after_attempt(5), wait=wait_exponential(min=1, max=10), retry=retry_if_exception_type(TransientHTTPError))
def gemini_call(system, user):
    r = requests.post(GEMINI_URL, timeout=60,
        headers={"Content-Type": "application/json", "x-api-key": API_KEY},
        json={
            "system_instruction": {"parts": [{"text": system}]},
            "contents": [{"role": "user", "parts": [{"text": user}]}],
            "generationConfig": {"temperature": TEMPERATURE, "maxOutputTokens": MAX_TOKENS}
        })
    _raise_for_status(r)
    return r.json()["candidates"][0]["content"]["parts"][0]["text"].strip()

# ==============================
# EGNA TWO-STAGE
# ==============================
def run_egna(prompt, model):
    # Stage I
    if model["type"] == "openai":
        stage_i = openai_call(STAGE_I_SYSTEM, prompt, model["id"])
    elif model["type"] == "claude":
        stage_i = claude_call(STAGE_I_SYSTEM, prompt)
    elif model["type"] == "gemini":
        stage_i = gemini_call(STAGE_I_SYSTEM, prompt)

    # Stage II
    stage_ii_system = STAGE_II_SYSTEM.format(stage_i=stage_i)
    if model["type"] == "openai":
        stage_ii = openai_call(stage_ii_system, prompt, model["id"])
    elif model["type"] == "claude":
        stage_ii = claude_call(stage_ii_system, prompt)
    elif model["type"] == "gemini":
        stage_ii = gemini_call(stage_ii_system, prompt)

    return stage_i, stage_ii

# ==============================
# RUN
# ==============================
all_results = {}

for model in MODELS:
    print(f"\nRunning: {model['name']}")
    rows = []
    for i, prompt in enumerate(PROMPTS, 1):
        print(f"  Prompt {i}/{len(PROMPTS)}")
        for iteration in range(1, ITERATIONS + 1):
            try:
                stage_i, stage_ii = run_egna(prompt, model)
            except Exception as e:
                stage_i = f"ERROR: {e}"
                stage_ii = f"ERROR: {e}"
            rows.append({
                "prompt_id": i,
                "prompt": prompt,
                "iteration": iteration,
                "stage_i": stage_i,
                "stage_ii": stage_ii
            })
        time.sleep(0.5)
    all_results[model["name"]] = pd.DataFrame(rows)
    print(f"  ✅ {model['name']} done")

with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
    for model_name, df in all_results.items():
        df.to_excel(writer, sheet_name=model_name[:31], index=False)

print(f"\n✅ Saved to {OUTPUT_FILE}")

from google.colab import files
files.download(OUTPUT_FILE)


Running: GPT-4o-mini
  Prompt 1/4
  Prompt 2/4
  Prompt 3/4
  Prompt 4/4
  ✅ GPT-4o-mini done

Running: GPT-4o
  Prompt 1/4
  Prompt 2/4
  Prompt 3/4
  Prompt 4/4
  ✅ GPT-4o done

Running: GPT-5.1
  Prompt 1/4
  Prompt 2/4
  Prompt 3/4
  Prompt 4/4
  ✅ GPT-5.1 done

Running: Gemini-2.0-Flash
  Prompt 1/4
  Prompt 2/4
  Prompt 3/4
  Prompt 4/4
  ✅ Gemini-2.0-Flash done

Running: Claude-Sonnet-4
  Prompt 1/4
  Prompt 2/4
  Prompt 3/4
  Prompt 4/4
  ✅ Claude-Sonnet-4 done

✅ Saved to fake_template_egna_results.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>